# 따릉이 스테이션 별 수요도 예측 Baseline

## 환경 설정

In [3]:
import sys
import os

from IPython.core.magics import display

sys.path.append(os.path.dirname(os.getcwd())) # 상위 폴더(프로젝트 루트)를 모듈 검색 경로에 추가

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import optuna

plt.rcParams["axes.unicode_minus"] = False # 그래프에서 음수 부호(-)가 깨지는 문제 방지
plt.rcParams['font.family'] = 'Malgun Gothic' # 한글 폰트 설정

SEED = 42 # 난수 시드 고정
np.random.seed(SEED)

print("라이브러리 로드 완료")

라이브러리 로드 완료


## 데이터베이스 연결 및 데이터 조회

In [4]:
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()
DB_USER     = os.getenv("DB_USER", "root")
DB_PASSWORD = os.getenv("DB_PASSWORD", "password")
DB_HOST     = os.getenv("DB_HOST", "localhost")
DB_PORT     = os.getenv("DB_PORT", "3306")
DB_NAME     = os.getenv("DB_NAME", "seoul_bike") # 첨부 이미지 참조

DATABASE_URL = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)

with engine.connect() as conn:
    print("데이터베이스 연결 성공" if conn.execute(text("SELECT 1")).scalar() == 1 else "데이터베이스 연결 실패")

hourly_air_2024_df = pd.read_sql_table("hourly_air_2024", con=engine)
hourly_precip_2024_df = pd.read_sql_table("hourly_precip_2024", con=engine)
hourly_snow_2024_df = pd.read_sql_table("hourly_snow_2024", con=engine)
hourly_temp_2024_df = pd.read_sql_table("hourly_temp_2024", con=engine)
infra_business_df = pd.read_sql_table("infra_business", con=engine)
infra_park_df = pd.read_sql_table("infra_park", con=engine)
infra_river_df = pd.read_sql_table("infra_river", con=engine)
infra_school_df = pd.read_sql_table("infra_school", con=engine)
infra_univ_df = pd.read_sql_table("infra_univ", con=engine)
korea_holidays_df = pd.read_sql_table("korea_holidays", con=engine)
pop_flow_2024_df = pd.read_sql_table("pop_flow_2024", con=engine)
pop_living_2024_df = pd.read_sql_table("pop_living_2024", con=engine)
rent_history_df = pd.read_sql_table("rent_history", con=engine)
rt_air_df = pd.read_sql_table("rt_air", con=engine)
rt_bike_status_df = pd.read_sql_table("rt_bike_status", con=engine)
rt_weather_df = pd.read_sql_table("rt_weather", con=engine)
station_loc_df = pd.read_sql_table("station_loc", con=engine)

데이터베이스 연결 성공


In [6]:
hourly_air_2024_df.info()
hourly_air_2024_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 420540 entries, 0 to 420539
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   id            420540 non-null  int64         
 1   measure_date  420540 non-null  datetime64[ns]
 2   region_name   420540 non-null  object        
 3   pm10          420540 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1), object(1)
memory usage: 12.8+ MB


,id,measure_date,region_name,pm10
0,1,2024-01-01 00:00:00,영등포구,19.0
1,2,2024-01-01 00:00:00,마포구,19.0
2,3,2024-01-01 00:00:00,송파구,19.0
3,4,2024-01-01 00:00:00,강서구,19.0
4,5,2024-01-01 00:05:00,영등포구,19.0


In [7]:
hourly_precip_2024_df.info()
hourly_precip_2024_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2908 entries, 0 to 2907
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   id             2908 non-null   int64         
 1   measure_date   2908 non-null   datetime64[ns]
 2   region_name    2908 non-null   object        
 3   precipitation  2908 non-null   float64       
dtypes: datetime64[ns](1), float64(1), int64(1), object(1)
memory usage: 91.0+ KB


,id,measure_date,region_name,precipitation
0,1,2024-01-01,영등포구,0.0
1,2,2024-01-02,영등포구,0.0
2,3,2024-01-03,영등포구,0.0
3,4,2024-01-04,영등포구,0.0
4,5,2024-01-05,영등포구,0.0


In [8]:
hourly_snow_2024_df.info()
hourly_snow_2024_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35064 entries, 0 to 35063
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   id            35064 non-null  int64         
 1   measure_date  35064 non-null  datetime64[ns]
 2   region_name   35064 non-null  object        
 3   snowfall      35064 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1), object(1)
memory usage: 1.1+ MB


,id,measure_date,region_name,snowfall
0,1,2024-01-01 00:00:00,영등포구,0.0
1,2,2024-01-01 00:00:00,마포구,0.0
2,3,2024-01-01 00:00:00,송파구,0.0
3,4,2024-01-01 00:00:00,강서구,0.0
4,5,2024-01-01 01:00:00,영등포구,0.0


## 전처리

## EDA

## 피처(X) / 타깃(Y) 분리

## Train / Validation / Test 3분할

## 평가 지표 함수 및 기본 모델 학습

## 여러 모델 비교(Ridge, RandomForest, XGBoost, LightGBM)

## 앙상블(Voting Regressor)

## 최종 모델 선택

## Test셋 최종 평가

## 모델 저장(pkl) 및 저장된 모델 검증